
# Step 2 — Data Curation (Production Version)

In this step we **load the dataset from `data/raw`** and clean it.

The main goals:

1. Load the raw dataset saved in Step 1
2. Parse salary text into numeric values
3. Remove rows where salary cannot be parsed
4. Remove salary outliers
5. Save the curated dataset to `data/processed`

This step is where **salary parsing happens**.


## 1. Load Raw Dataset

In [ ]:

import json
import pandas as pd

with open("../data/raw/jobs_raw.json") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)

print("Rows loaded:", len(df))
df.head()



## 2. Salary Parsing

The salary column currently contains **text** like:

```
"$120,000 - $150,000 a year"
"$60 - $80 an hour"
```

We convert this into a **single numeric salary value**.

Example:

```
"$120,000 - $150,000"
↓
135000
```


In [ ]:

import re

def parse_salary(text):

    if text is None:
        return None

    # extract all numbers
    nums = re.findall(r"[0-9,]+", text)

    if not nums:
        return None

    nums = [int(n.replace(",", "")) for n in nums]

    # average salary range
    return sum(nums) / len(nums)


## 3. Apply Salary Parsing

In [ ]:

df["salary_parsed"] = df["salary"].apply(parse_salary)

df[["salary","salary_parsed"]].head(10)



## 4. Remove Rows With Missing Salaries


In [ ]:

df = df[df.salary_parsed.notna()]

print("Remaining rows:", len(df))



## 5. Remove Salary Outliers

We remove unrealistic values:

- below 20,000
- above 500,000


In [ ]:

df = df[(df.salary_parsed > 20000) & (df.salary_parsed < 500000)]

print("Dataset size after cleaning:", len(df))



## 6. Preview Curated Dataset


In [ ]:

df[["positionName","location","salary_parsed"]].head()



## 7. Save Curated Dataset

This dataset will be used in **Step 3 (LLM Extraction)**.


In [ ]:

import os

os.makedirs("../data/processed", exist_ok=True)

df.to_json("../data/processed/jobs_curated.json", orient="records")

print("Saved curated dataset to data/processed/jobs_curated.json")
